# Public Rainfall Distribution Fitting Notebook

This notebook shows how to fit rainfall distributions for any area of interest using CHIRPS daily NetCDF files. It is a public, simplified version of the private workflow and keeps only the core fitting steps.

It is written so the user can see every required input file and every output artifact produced by the workflow.

## Inputs

- `RWH_AOI_PATH`: AOI boundary file, such as a shapefile or GeoPackage.
- `RWH_CHIRPS_DIR`: Folder containing CHIRPS daily NetCDF files.
- `RWH_OUTPUT_DIR`: Folder where all outputs are written.
- Optional: `RWH_PUBLIC_BASE` to change the base directory for default paths.

## Outputs

- `cache/daily_rainfall_cube.pkl`: cached rainfall cube used to speed up reruns.
- `tables/aoi_pixels.csv`: CHIRPS pixel locations inside the AOI.
- `tables/rainfall_distribution_summary.csv`: best-fit row per pixel.
- `tables/rainfall_distribution_details.csv`: all candidate fits per pixel.
- `tables/run_metadata.json`: run configuration and file counts.
- `plots/sample_fit.png`: an example histogram and fitted density curves.

## Coverage

- Full candidate list from the private workflow is included in `fit_distributions`.
- The notebook exports per-pixel fit metrics and a detailed table of all candidate distributions.
- The sample plot uses the top-ranked candidates for one pixel so users can see how the fits compare.

## Workflow

1. Load the AOI boundary.
2. Find CHIRPS grid cells inside the AOI.
3. Load or build the rainfall cube.
4. Fit standard distributions pixel by pixel.
5. Save the summary table, detailed table, metadata, and example plot.

In [ ]:
# 1. Imports and configuration
from pathlib import Path
import os
import json
import pickle

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from shapely.geometry import Point
import scipy.stats as stats
from scipy.stats import rv_continuous, kstest, chisquare
from scipy.special import beta as beta_fn, betainc, betaincinv
import matplotlib.pyplot as plt

BASE_DIR = Path(os.environ.get('RWH_PUBLIC_BASE', r'c:/CHRISP/Regional_RWH')).resolve()
AOI_PATH = Path(os.environ.get('RWH_AOI_PATH', str(BASE_DIR / 'data' / 'aoi.shp')))
CHIRPS_DIR = Path(os.environ.get('RWH_CHIRPS_DIR', str(BASE_DIR / 'data' / 'chirps_daily')))
OUTPUT_DIR = Path(os.environ.get('RWH_OUTPUT_DIR', str(BASE_DIR / 'output_public')))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = OUTPUT_DIR / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR = OUTPUT_DIR / 'tables'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR = OUTPUT_DIR / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

RAIN_VAR_CANDIDATES = ['precip', 'precipitation', 'rain']
QUANTILES = [0.10, 0.50, 0.90]
MIN_WET_DAYS = 365

RUN_METADATA_PATH = TABLE_DIR / 'run_metadata.json'
AOI_PIXELS_PATH = TABLE_DIR / 'aoi_pixels.csv'
SUMMARY_CSV_PATH = TABLE_DIR / 'rainfall_distribution_summary.csv'
DETAILS_CSV_PATH = TABLE_DIR / 'rainfall_distribution_details.csv'
SAMPLE_PLOT_PATH = PLOTS_DIR / 'sample_fit.png'
CACHE_PATH = CACHE_DIR / 'daily_rainfall_cube.pkl'

# Full candidate set requested for public reuse.
fit_distributions = [
    'lognorm', 'gamma', 'gengamma', 'gb2', 'betaprime', 'burr', 'burr12', 'dagum', 'lomax', 'fisk',
    'genpareto', 'genlogistic', 'loggamma', 'loglaplace', 'halfnorm', 'levy', 'gumbel_r', 'genextreme',
    'norm', 'pareto', 'pearson3', 'weibull_min', 'invweibull', 'expon', 'exponweibull', 'genexpon', 'geninvgauss',
    'invgauss', 'fatiguelife', 'gennorm', 'kappa4', 'skewnorm'
]

# Parameter hints for positive-support distributions.
distribution_fit_kwargs = {
    'gb2': {'floc': 0},
    'dagum': {'floc': 0},
    'lognorm': {'floc': 0},
    'gamma': {'floc': 0},
    'gengamma': {'floc': 0},
    'betaprime': {'floc': 0},
    'burr': {'floc': 0},
    'burr12': {'floc': 0},
    'lomax': {'floc': 0},
    'fisk': {'floc': 0},
    'genpareto': {'floc': 0},
    'loggamma': {'floc': 0},
    'loglaplace': {'floc': 0},
    'weibull_min': {'floc': 0},
    'invweibull': {'floc': 0},
    'expon': {'floc': 0},
    'exponweibull': {'floc': 0},
    'genexpon': {'floc': 0},
    'geninvgauss': {'floc': 0},
    'invgauss': {'floc': 0},
    'fatiguelife': {'floc': 0},
    'pareto': {'floc': 0},
    'pearson3': {'floc': 0}
}

class gb2_gen(rv_continuous):
    shapes = 'alpha, p, q'

    def __init__(self, *args, **kwargs):
        kwargs.setdefault('a', 0.0)
        super().__init__(*args, **kwargs)

    def _argcheck(self, alpha, p, q):
        return (alpha > 0) & (p > 0) & (q > 0)

    def _pdf(self, x, alpha, p, q):
        x = np.asarray(x, dtype=float)
        pdf = np.zeros_like(x, dtype=float)
        mask = x > 0
        if np.any(mask):
            xa = np.power(x[mask], alpha)
            numerator = alpha * np.power(x[mask], alpha * p - 1.0)
            denominator = beta_fn(p, q) * np.power(1.0 + xa, p + q)
            pdf[mask] = numerator / denominator
        return pdf

    def _cdf(self, x, alpha, p, q):
        x = np.asarray(x, dtype=float)
        cdf = np.zeros_like(x, dtype=float)
        mask = x > 0
        if np.any(mask):
            xa = np.power(x[mask], alpha)
            z = np.clip(xa / (1.0 + xa), 0.0, 1.0)
            cdf[mask] = betainc(p, q, z)
        return cdf

    def _ppf(self, probabilities, alpha, p, q):
        probabilities = np.asarray(probabilities, dtype=float)
        result = np.zeros_like(probabilities, dtype=float)
        lower = probabilities <= 0.0
        upper = probabilities >= 1.0
        mid = (~lower) & (~upper)
        result[lower] = 0.0
        if np.any(mid):
            z = betaincinv(p, q, np.clip(probabilities[mid], 1e-12, 1 - 1e-12))
            ratio = np.clip(z / (1.0 - z), 0.0, None)
            result[mid] = np.power(ratio, 1.0 / alpha)
        result[upper] = np.inf
        return result

class dagum_gen(rv_continuous):
    shapes = 'alpha, p'

    def __init__(self, *args, **kwargs):
        kwargs.setdefault('a', 0.0)
        super().__init__(*args, **kwargs)

    def _argcheck(self, alpha, p):
        return (alpha > 0) & (p > 0)

    def _pdf(self, x, alpha, p):
        x = np.asarray(x, dtype=float)
        pdf = np.zeros_like(x, dtype=float)
        mask = x > 0
        if np.any(mask):
            xa = np.power(x[mask], alpha)
            pdf[mask] = (alpha * p * np.power(x[mask], alpha * p - 1.0)) / np.power(1.0 + xa, p + 1.0)
        return pdf

    def _cdf(self, x, alpha, p):
        x = np.asarray(x, dtype=float)
        cdf = np.zeros_like(x, dtype=float)
        mask = x > 0
        if np.any(mask):
            cdf[mask] = np.power(1.0 + np.power(x[mask], -alpha), -p)
        return cdf

    def _ppf(self, probabilities, alpha, p):
        probabilities = np.asarray(probabilities, dtype=float)
        result = np.zeros_like(probabilities, dtype=float)
        lower = probabilities <= 0.0
        upper = probabilities >= 1.0
        mid = (~lower) & (~upper)
        result[lower] = 0.0
        if np.any(mid):
            ratio = np.power(np.clip(probabilities[mid], 1e-12, 1 - 1e-12), -1.0 / p) - 1.0
            ratio = np.clip(ratio, 1e-12, None)
            result[mid] = np.power(ratio, -1.0 / alpha)
        result[upper] = np.inf
        return result


gb2_distribution = gb2_gen(name='gb2', a=0.0)
dagum_distribution = dagum_gen(name='dagum', a=0.0)

# Aliases for public use when SciPy uses a slightly different name.
distribution_registry = {
    'gb2': gb2_distribution,
    'dagum': dagum_distribution,
    'exponweibull': stats.exponweib,
}

print('Configuration ready.')
print(f'AOI_PATH = {AOI_PATH}')
print(f'CHIRPS_DIR = {CHIRPS_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

In [ ]:
# 2. Load AOI and find CHIRPS pixels inside it
if not AOI_PATH.exists():
    raise FileNotFoundError(f'Boundary file not found: {AOI_PATH}')

chirps_files = sorted(CHIRPS_DIR.glob('*.nc'))
if not chirps_files:

aoi_gdf = gpd.read_file(AOI_PATH)
if aoi_gdf.crs is None:
    aoi_gdf = aoi_gdf.set_crs('EPSG:4326')
else:
    aoi_gdf = aoi_gdf.to_crs('EPSG:4326')
aoi_boundary = aoi_gdf.dissolve()

with xr.open_dataset(chirps_files[0]) as sample_ds:
    if 'lat' in sample_ds.coords and 'lon' in sample_ds.coords:
        lat_name, lon_name = 'lat', 'lon'
    elif 'latitude' in sample_ds.coords and 'longitude' in sample_ds.coords:
        lat_name, lon_name = 'latitude', 'longitude'
    elif 'y' in sample_ds.coords and 'x' in sample_ds.coords:
        lat_name, lon_name = 'y', 'x'
    else:
        raise KeyError(f'Could not detect coordinate names: {list(sample_ds.coords.keys())}')

    rain_var = next((name for name in RAIN_VAR_CANDIDATES if name in sample_ds.data_vars), None)
    if rain_var is None:
        raise KeyError(f'Could not find a rainfall variable in {chirps_files[0].name}')

    lat_vals = sample_ds[lat_name].values
    lon_vals = sample_ds[lon_name].values

lon_grid, lat_grid = np.meshgrid(lon_vals, lat_vals)
grid_points = gpd.GeoDataFrame({'geometry': [Point(lon, lat) for lon, lat in zip(lon_grid.ravel(), lat_grid.ravel())]}, crs='EPSG:4326')
grid_points['lat'] = grid_points.geometry.y
grid_points['lon'] = grid_points.geometry.x
pixels_in_aoi = gpd.sjoin(grid_points, aoi_boundary, how='inner', predicate='within').drop_duplicates(subset=['lat', 'lon']).reset_index(drop=True)
pixels_in_aoi[['lat', 'lon']].to_csv(AOI_PIXELS_PATH, index=False)

print(f'Found {len(pixels_in_aoi)} CHIRPS pixels inside the AOI.')
print(f'Saved pixel list to {AOI_PIXELS_PATH}')

In [ ]:
# 3. Build or load the rainfall cube
if CACHE_PATH.exists():
    with open(CACHE_PATH, 'rb') as f:
        daily_all = pickle.load(f)
    print(f'Loaded rainfall cube from cache: {CACHE_PATH}')
else:
    daily_list = []
    lat_min, lat_max = pixels_in_aoi['lat'].min(), pixels_in_aoi['lat'].max()
    lon_min, lon_max = pixels_in_aoi['lon'].min(), pixels_in_aoi['lon'].max()

    for fp in chirps_files:
        with xr.open_dataset(fp) as ds:
            if 'lat' in ds.coords and 'lon' in ds.coords:
                lat_n, lon_n = 'lat', 'lon'
            elif 'latitude' in ds.coords and 'longitude' in ds.coords:
                lat_n, lon_n = 'latitude', 'longitude'
            elif 'y' in ds.coords and 'x' in ds.coords:
                lat_n, lon_n = 'y', 'x'
            else:
                continue

            rain_var = next((name for name in RAIN_VAR_CANDIDATES if name in ds.data_vars), None)
            if rain_var is None:
                continue

            ds_r = ds.rename({lat_n: 'lat', lon_n: 'lon'})
            lat_vals = ds_r['lat'].values
            lon_vals = ds_r['lon'].values
            lat_slice = slice(lat_min, lat_max) if lat_vals[0] <= lat_vals[-1] else slice(lat_max, lat_min)
            lon_slice = slice(lon_min, lon_max) if lon_vals[0] <= lon_vals[-1] else slice(lon_max, lon_min)
            crop = ds_r[rain_var].sel(lat=lat_slice, lon=lon_slice).compute()
            daily_list.append(crop)

    if not daily_list:
        raise RuntimeError('No rainfall data could be loaded.')

    daily_all = xr.concat(daily_list, dim='time').sortby('time')
    with open(CACHE_PATH, 'wb') as f:
        pickle.dump(daily_all, f)
    print(f'Cached rainfall cube to {CACHE_PATH}')

print(daily_all)

In [ ]:
# 4. Fit distributions and save the best result for each pixel
results = []
detail_rows = []
quantile_probs = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
quantile_labels = [f'p{int(prob * 100):02d}' for prob in quantile_probs]

for _, row in pixels_in_aoi[['lat', 'lon']].drop_duplicates().iterrows():
    lat = float(row['lat'])
    lon = float(row['lon'])

    try:
        series = daily_all.sel(lat=lat, lon=lon, method='nearest').to_dataframe(name='rain_mm').reset_index()
    except Exception:
        continue

    data = series['rain_mm'].dropna().astype(float).values
    data = data[data >= 1.0]
    if data.size < MIN_WET_DAYS:
        continue

    fit_rows = []
    for dist_name in fit_distributions:
        dist = distribution_registry.get(dist_name, getattr(stats, dist_name, None))
        if dist is None:
            continue
        fit_kwargs = distribution_fit_kwargs.get(dist_name, {})
        try:
            params = dist.fit(data, **fit_kwargs)
            log_likelihood = np.sum(dist.logpdf(data, *params))
            aic = 2 * len(params) - 2 * log_likelihood
            quantiles = dist.ppf(quantile_probs, *params)
            fit_row = {
                'lat': lat,
                'lon': lon,
                'distribution': dist_name,
                'params': list(map(float, params)),
                'aic': float(aic),
                'log_likelihood': float(log_likelihood),
                'p10': float(quantiles[0]),
                'p25': float(quantiles[1]),
                'p50': float(quantiles[2]),
                'p75': float(quantiles[3]),
                'p90': float(quantiles[4]),
                'p95': float(quantiles[5]),
                'p99': float(quantiles[6]),
                'n_wet_days': int(data.size),
                'mean_rain_mm': float(np.mean(data)),
                'median_rain_mm': float(np.median(data)),
                'std_rain_mm': float(np.std(data))
            }
            fit_rows.append(fit_row)
            detail_rows.append(fit_row)
        except Exception:
            continue

    if not fit_rows:
        continue

    fit_df = pd.DataFrame(fit_rows).sort_values('aic').reset_index(drop=True)
    fit_df['delta_aic'] = fit_df['aic'] - fit_df['aic'].min()
    with np.errstate(over='ignore'):
        weights = np.exp(-0.5 * fit_df['delta_aic'].to_numpy(dtype=float))
    weight_sum = float(np.nansum(weights))
    fit_df['aic_weight'] = weights / weight_sum if np.isfinite(weight_sum) and weight_sum > 0 else np.nan
    fit_df['rank_aic'] = np.arange(1, len(fit_df) + 1)

    best = fit_df.iloc[0].to_dict()
    best['top_5_distributions'] = fit_df.head(5)[['distribution', 'aic', 'delta_aic', 'aic_weight']].to_dict('records')
    results.append(best)

results_df = pd.DataFrame(results)
details_df = pd.DataFrame(detail_rows)
results_df.to_csv(SUMMARY_CSV_PATH, index=False)
details_df.to_csv(DETAILS_CSV_PATH, index=False)

run_metadata = {
    'aoi_path': str(AOI_PATH),
    'chirps_dir': str(CHIRPS_DIR),
    'output_dir': str(OUTPUT_DIR),
    'aoi_pixel_count': int(len(pixels_in_aoi)),
    'summary_rows': int(len(results_df)),
    'detail_rows': int(len(details_df)),
    'chirps_file_count': int(len(chirps_files)),
    'distributions': fit_distributions,
    'quantiles': quantile_labels,
    'min_wet_days': MIN_WET_DAYS
}
with open(RUN_METADATA_PATH, 'w', encoding='utf-8') as f:
    json.dump(run_metadata, f, indent=2)

print(f'Saved summary table to {SUMMARY_CSV_PATH}')
print(f'Saved detail table to {DETAILS_CSV_PATH}')
print(f'Saved run metadata to {RUN_METADATA_PATH}')
results_df.head()

In [ ]:
# 5. Plot one example fit
if results_df.empty:
    print('No fitted results to plot.')
else:
    sample = results_df.iloc[0]
    sample_series = daily_all.sel(lat=float(sample['lat']), lon=float(sample['lon']), method='nearest').to_dataframe(name='rain_mm').reset_index()['rain_mm'].dropna().astype(float).values
    sample_series = sample_series[sample_series >= 1.0]
    x = np.linspace(sample_series.min(), sample_series.max(), 200)

    plt.figure(figsize=(10, 6))
    plt.hist(sample_series, bins=30, density=True, alpha=0.4, color='gray', label='Observed')

    top_5 = sample.get('top_5_distributions', [])
    if not top_5:
        top_5 = results_df[results_df['lat'] == float(sample['lat'])]
        top_5 = top_5[top_5['lon'] == float(sample['lon'])].sort_values('aic').head(5).to_dict('records')

    for row in top_5:
        dist_name = row.get('distribution')
        dist = distribution_registry.get(dist_name, getattr(stats, dist_name, None))
        if dist is None:
            continue
        params = row.get('params', [])
        try:
            y = dist.pdf(x, *params)
            label = f"{dist_name} (AIC: {float(row.get('aic', np.nan)):.1f})"
            plt.plot(x, y, label=label)
        except Exception:
            continue

    plt.legend()
    plt.xlabel('Daily rainfall (mm)')
    plt.ylabel('Density')
    plt.title('Sample rainfall distribution fit')
    plt.tight_layout()
    plt.savefig(SAMPLE_PLOT_PATH, dpi=150)
    plt.close()
    print(f'Saved sample plot to {SAMPLE_PLOT_PATH}')

## Notes

- Change `RWH_AOI_PATH` to any shapefile or GeoPackage for another country or region.
- Change `RWH_CHIRPS_DIR` to your CHIRPS NetCDF folder.
- The notebook writes all major inputs and outputs to the folders listed above so it can be reused and shared publicly.